# LightGBM — Network Intrusion Detection

Huấn luyện và đánh giá mô hình **LightGBM** trên tập dữ liệu CICIDS2017.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb

# Thêm thư mục gốc dự án vào sys.path
sys.path.insert(0, str(Path.cwd()))

from src.model_training import (
    load_splits,
    evaluate_model,
    plot_confusion_matrix,
    compare_models,
)

In [ ]:
# --- STEP 1: Load dữ liệu đã chia sẵn ---
print("=" * 70)
print("STEP 1: Loading pre-split data")
print("=" * 70)

X_train, X_test, y_train, y_test = load_splits()

print(f"Feature columns: {X_train.shape[1]}")
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Classes: {sorted(y_train.unique())}")

In [ ]:
# --- STEP 2: Train LightGBM ---
print("\n" + "=" * 70)
print("STEP 2: Training LightGBM (n_estimators=200)")
print("=" * 70)

t0 = time.time()

lgb_model = lgb.LGBMClassifier(
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

lgb_model.fit(X_train, y_train)

train_time = time.time() - t0
print(f"Training completed in {train_time:.2f} seconds")
print(f"Number of features: {lgb_model.n_features_in_}")
print(f"Number of classes: {len(lgb_model.classes_)}")

In [ ]:
# --- STEP 3: Evaluate ---
print("\n" + "=" * 70)
print("STEP 3: Model Evaluation")
print("=" * 70)

y_pred = lgb_model.predict(X_test)
y_pred_proba = lgb_model.predict_proba(X_test)

lgb_results = evaluate_model(
    y_true=y_test,
    y_pred=y_pred,
    model_name="LightGBM (n_estimators=200)",
    y_pred_proba=y_pred_proba,
    labels=lgb_model.classes_.tolist(),
)

In [ ]:
# --- STEP 4: Confusion Matrix ---
print("\n" + "=" * 70)
print("STEP 4: Confusion Matrix")
print("=" * 70)

plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred,
    labels=lgb_model.classes_.tolist(),
    model_name="LightGBM",
    normalize=True,
    figsize=(12, 10),
)

In [ ]:
# --- STEP 5: Feature Importance ---
print("\n" + "=" * 70)
print("STEP 5: Feature Importance (Top 20)")
print("=" * 70)

feat_imp = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": lgb_model.feature_importances_,
}).sort_values("Importance", ascending=False)

print(feat_imp.head(20).to_string(index=False))

plt.figure(figsize=(12, 6))
sns.barplot(
    data=feat_imp.head(20),
    x="Importance", y="Feature",
    hue="Feature", legend=False,
    palette="viridis",
)
plt.title("Top 20 Feature Importance — LightGBM")
plt.xlabel("Importance Score")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# --- STEP 6: Summary ---
print("\n" + "=" * 70)
print("STEP 6: Summary")
print("=" * 70)

comparison_df = compare_models([lgb_results])
print("\nPerformance Metrics:")
print(comparison_df.to_string(index=False))

print(f"\nTraining time  : {train_time:.2f}s")
print(f"Test set size  : {len(y_test):,}")
print(f"Estimators     : 200")
print(f"Features       : {X_train.shape[1]}")
print(f"Classes        : {len(lgb_model.classes_)}")